# Lab 9.2 &mdash; Cite Every Claim

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Build a claim that carries its statement, value AND the exact source string
- Assemble a multi-claim summary and detect which claims are uncited
- See why one uncited claim in a mix breaks auditability

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; grounding/citation/compute logic and, in the agent-assembly labs, tool wiring &amp; the read-only guardrail &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It grounds &amp; cites every figure, gives **no advice**, and has **no trade tool** &mdash; a human analyst decides.

**Reference:** [Module 9 slides &mdash; Auditability: structure & the trail](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-09-02"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Lab 1 checked a *single* figure was grounded. A real summary makes **many** claims at once, and the
danger is the **mix**: five cited numbers and one silently uncited one. Auditability means every
conclusion is **traceable** (deck slide 15), so each **claim** carries the exact **source string**
it came from &mdash; the citation a later validation step (Lab 7) checks for *correctness*. Here you
build the claim record and a detector that names **which** claims in a summary are uncited.

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

print("a claim carries the SOURCE STRING through, e.g.:")
print({"statement": "revenue", "value": 120.0, "source": "p4, income stmt"})
print("a summary is a LIST of these -- and we must find any that are uncited.")

## Your Turn
Complete `make_claim` (carry the source string through) and `uncited_claims` (return the statements of
every claim in a summary that is missing a citation &mdash; the mix detector).

In [ ]:
def make_claim(statement, fig):
    # a claim ties a statement to its grounded value AND carries its exact source string
    return {"statement": statement, "value": fig["value"], "source": ___}   # TODO: the citation

def uncited_claims(claims):
    # return the STATEMENT of each claim in the summary that is missing a source citation
    return [___ for c in claims if not c.get("source")]   # TODO: the statement of each uncited claim

try:
    c = make_claim('revenue', REPORT['revenue'])
    print('claim:', c)
    summary = [make_claim('revenue', REPORT['revenue']),
               make_claim('net_income', REPORT['net_income']),
               {'statement': 'guess', 'value': 5.0, 'source': ''}]   # a slipped-in uncited claim
    print('uncited in the mix:', uncited_claims(summary))
    print('fully-cited pair  :', uncited_claims(summary[:2]))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a claim carries its value", lambda: make_claim("revenue", REPORT["revenue"])["value"] == 120.0)
expect_true("a claim carries the exact source string through", lambda: make_claim("net_income", REPORT["net_income"])["source"] == "p4, income stmt")
expect_true("a fully-cited summary has no uncited claims", lambda: uncited_claims([make_claim("revenue", REPORT["revenue"]), make_claim("net_income", REPORT["net_income"])]) == [])
expect_true("the mix detector names the one uncited claim", lambda: uncited_claims([make_claim("revenue", REPORT["revenue"]), {"statement": "guess", "value": 5.0, "source": ""}]) == ["guess"])
expect_true("several uncited claims are all listed", lambda: set(uncited_claims([{"statement": "a", "source": ""}, {"statement": "b", "source": None}, make_claim("revenue", REPORT["revenue"])])) == {"a", "b"})

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
A summary is a mix of claims, and one silently uncited number breaks the chain. Carrying the exact source string through -- and naming which claims lack it -- is what a validator (Lab 7) checks for correctness and what makes the agent auditable.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>